# How good are our topic labels?

Our T5 model (`models/attempt7-t5-leave-keywords-in-epoch-1`) labeled every ACLED event
(`data/filtered_events_class_with_predicted.csv`). This notebook follows the same three steps as `verify_students.ipynb`:

1. **Score** the labels against 200 manually labeled random `unknown` events.
2. **Review** every error by hand. Is the prediction really wrong, or is it defensible?
3. **Explain** the real errors. Each issue is tested on all ~84,000 unknown events, and the key examples are
   re-run with words removed to confirm the cause.

The last step needs the model checkpoint. It runs on CPU in about a minute.

In [1]:
import os, re
import pandas as pd
pd.set_option("display.max_colwidth", 120)

MANUAL_CSV = "../data/manual_labelled_data/random_unknown_labeled.csv"
OURS_CSV   = "../data/filtered_events_class_with_predicted.csv"
TRAIN_CSV  = "../data/labeled_balanced_20.csv"
REVIEW_CSV = "ours_error_review.csv"
MODEL_DIR  = "../models/attempt7-t5-leave-keywords-in-epoch-1/hf_transformer_model"

ours = pd.read_csv(OURS_CSV, usecols=["event_id_cnty", "event_date", "notes", "clean_notes", "class", "predicted_class"],
                   low_memory=False)
ours["date"] = pd.to_datetime(ours["event_date"])
unknown = ours[ours["class"] == "unknown"]
text = unknown["clean_notes"].fillna("")

ev = pd.read_csv(MANUAL_CSV, usecols=["event_id_cnty", "manual_label", "manual_label_alt", "predicted_class_students", "notes"])
ev = ev.merge(ours[["event_id_cnty", "date", "clean_notes", "predicted_class"]], on="event_id_cnty", how="left")
assert ev["predicted_class"].notna().all()
print(f"{len(ev)} manually labeled events, {len(unknown):,} unknown events in our file")

200 manually labeled events, 83,921 unknown events in our file


## 1. Score against the manual labels

Same rules as for the students. **Strict**: the prediction equals the manual label.
**Lenient**: the prediction equals the manual label or its alternative, or forms one of the accepted pairs.

In [2]:
ACCEPTED_PAIRS = {frozenset(p) for p in [
    ("climate", "environment"),
    ("unjust law enforcement", "blm"),
    ("public services", "health care"), ("public services", "education"), ("public services", "housing"),
    ("discrimination", "women rights"), ("discrimination", "lgbtq"),      ("discrimination", "blm"),
]}

def gold_labels(row):
    golds = {row["manual_label"]}
    if isinstance(row["manual_label_alt"], str):
        golds |= {x.strip() for x in row["manual_label_alt"].split("|")}
    return golds

def lenient(pred_col):
    return [any(g == p or frozenset((g, p)) in ACCEPTED_PAIRS for g in gold_labels(r))
            for (_, r), p in zip(ev.iterrows(), ev[pred_col])]

ev["strict"]  = ev["predicted_class"] == ev["manual_label"]
ev["lenient"] = lenient("predicted_class")
ev["students_lenient"] = lenient("predicted_class_students")

pd.DataFrame({
    "ours":     {"strict": ev["strict"].mean(), "lenient": ev["lenient"].mean()},
    "students": {"strict": (ev["predicted_class_students"] == ev["manual_label"]).mean(), "lenient": ev["students_lenient"].mean()},
}).style.format("{:.1%}")

,ours,students
strict,57.5%,55.0%
lenient,69.0%,68.5%


In [3]:
(ev.groupby("manual_label")
   .agg(events=("lenient", "size"), ours=("lenient", "mean"), students=("students_lenient", "mean"))
   .sort_values("events", ascending=False)
   .style.format({"ours": "{:.0%}", "students": "{:.0%}"}))

,events,ours,students
manual_label,,,
policies & politics,65,71%,69%
labor rights,50,52%,58%
public services,17,88%,82%
environment,16,69%,69%
climate,13,85%,69%
education,8,88%,100%
immigration,5,40%,40%
unjust law enforcement,5,80%,40%
health care,4,75%,50%


## 2. How wrong are the errors really?

Every event that fails lenient scoring was read and given one verdict (`ours_error_review.csv`), with the same
categories as for the students: **wrong**, **defensible**, or **no fitting class**.

In [4]:
review = pd.read_csv(REVIEW_CSV)
errors = ev[~ev["lenient"]].merge(review, on="event_id_cnty", how="left")
assert errors["verdict"].notna().all() and len(errors) == len(review), "review file out of sync with the errors"

print(errors["verdict"].value_counts().to_string(), "\n")

fits    = len(ev) - (errors["verdict"] == "no fitting class").sum()
correct = ev["lenient"].sum() + (errors["verdict"] == "defensible").sum()
print(f"Lenient accuracy:                                        {ev['lenient'].mean():.1%}")
print(f"Counting defensible as right, excluding no-fitting-class: {correct / fits:.1%}  ({correct}/{fits})")

verdict
wrong               37
defensible          16
no fitting class     9 

Lenient accuracy:                                        69.0%
Counting defensible as right, excluding no-fitting-class: 80.6%  (154/191)


In [5]:
with pd.option_context("display.max_rows", None):
    display(errors.sort_values(["verdict", "manual_label"])
                  [["event_id_cnty", "manual_label", "predicted_class", "verdict", "reason"]]
                  .set_index("event_id_cnty"))

,manual_label,predicted_class,verdict,reason
event_id_cnty,,,,
DEU727,culture,public services,defensible,closure of a municipal musical theater; public services is a fair reading
GBR3135,culture,policies & politics,defensible,objection to a town council planning decision
DEU22891,education,policies & politics,defensible,church budget cuts to student congregations; education is only loosely right
FRA27718,environment,policies & politics,defensible,protest is about government inaction after a factory fire
DEU8511,environment,public services,defensible,preserving a railway embankment; public services is a fair reading
BGR624,farmers,animal welfare,defensible,pig owners against a cull; animal welfare is a fair secondary reading
DEU1148,immigration,housing,defensible,about a refugee accommodation; housing is a partial reading
ESP8792,immigration,policies & politics,defensible,demand to reopen national borders is a policy demand
ROU2691,labor rights,discrimination,defensible,the note itself calls the pay exclusion discriminatory


### Same errors as the students?

Most errors are shared, so the two models mostly fail on the same hard events.

In [6]:
pd.crosstab(ev["lenient"].map({True: "ours right", False: "ours wrong"}),
            ev["students_lenient"].map({True: "students right", False: "students wrong"}))

students_lenient,students right,students wrong
lenient,,
ours right,118,20
ours wrong,19,43


## 3. Tools for the analysis

Our training file is exactly the first 3000 keyword-labeled rows per class of the date-sorted `labeled.csv`,
so every training row can be dated. `ablate` re-runs our model on a note with some words removed.

In [7]:
train = pd.read_csv(TRAIN_CSV).merge(
    ours[["clean_notes", "class", "date"]].drop_duplicates(["clean_notes", "class"]), on=["clean_notes", "class"], how="left")
assert train["date"].notna().all()

def word_effect(pattern, cls):
    """Share of `cls` among training rows with the word, and among unknown predictions with/without it."""
    in_train = train["clean_notes"].str.contains(pattern, regex=True, na=False)
    has = text.str.contains(pattern, regex=True)
    return {"word": pattern.replace("\\b", ""), "class": cls,
            "training rows with word": in_train.sum(),
            "training: share of class": (train.loc[in_train, "class"] == cls).mean(),
            "unknown: predicted with word": (unknown.loc[has, "predicted_class"] == cls).mean(),
            "unknown: predicted without": (unknown.loc[~has, "predicted_class"] == cls).mean()}

PCT = {c: "{:.0%}" for c in ["training: share of class", "unknown: predicted with word", "unknown: predicted without"]}

os.environ.setdefault("HF_HUB_OFFLINE", "1")
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval()   # CPU is fine for a few notes
id2label = {int(k): v for k, v in model.config.id2label.items()}

def ablate(event_id, *patterns):
    """Top-2 predictions for the full note and for the note with each pattern removed."""
    note = ev.set_index("event_id_cnty").loc[event_id, "clean_notes"]
    variants = {"full note": note} | {f"without '{p}'": re.sub(r"\s+", " ", re.sub(p, " ", note)).strip() for p in patterns}
    batch = tokenizer(list(variants.values()), return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.inference_mode():
        probs = model(**batch).logits.float().softmax(-1)
    rows = []
    for name, p in zip(variants, probs):
        v, i = p.topk(2)
        rows.append({"event": event_id, "input": name, "prediction": id2label[int(i[0])], "p": float(v[0]),
                     "runner-up": f"{id2label[int(i[1])]} {float(v[1]):.2f}"})
    return pd.DataFrame(rows)

def ablations(cases):
    return pd.concat([ablate(e, *p) for e, p in cases]).set_index(["event", "input"]).style.format({"p": "{:.2f}"})

/home/martan/Documents/personal/Nienke/Protest_Labelling/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/261 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 261/261 [00:00<00:00, 22722.27it/s]

## 4. Issue: energy prices are read as climate

Six errors involve energy or fuel prices: workers, fishers and citizens protesting the 2021–2022 price crisis.
Our model calls all six `climate` or `environment`.

In [8]:
ablations([("ROU1803", [r"\benergy\b"]), ("DEU13354", [r"\benergy\b"]), ("BEL1514", [r"\benergy\b"]),
           ("MDA1379", [r"\benergy\b"]), ("ITA14966", [r"\boil\b|\bgas\b"])])

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


In [9]:
pd.DataFrame([word_effect(r"\benergy\b", "climate"),
              word_effect(r"\bfuel\b|\bdiesel\b|\boil\b|\bgas\b", "climate")]).set_index("word").style.format(PCT)

,class,training rows with word,training: share of class,unknown: predicted with word,unknown: predicted without
word,,,,,
energy,climate,472,8%,58%,3%
fuel|diesel|oil|gas,climate,1364,32%,38%,3%


Removing the energy or fuel words moves every one of these events away from `climate`. Across all unknown events, a note mentioning
"energy" is labeled `climate` 58% of the time, against 3% without.

Label counts in the training data do not explain this, since only 8% of training rows with "energy" are climate.
The model seems to link energy, oil and gas to climate by itself, probably through pre-training and the climate
trigger "fossil fuels". The training rows about price crises also sit in the wrong classes for these events. They
are almost all farmers or Ukraine-war protests, with nearly none under `labor rights` or `policies & politics`:

In [10]:
crisis = train["clean_notes"].str.contains(r"\benergy price|\bprice energy|\bfuel price|\bprice fuel|cost living", regex=True)
print(f"training rows about energy/fuel prices or cost of living: {crisis.sum()} of {len(train):,}")
print(train.loc[crisis, "class"].value_counts().to_dict())

training rows about energy/fuel prices or cost of living: 176 of 49,765
{'farmers': 94, 'ukraine-russia war': 51, 'public services': 5, 'education': 4, 'housing': 3, 'immigration': 3, 'health care': 3, 'climate': 2, 'policies & politics': 2, 'environment': 2, 'discrimination': 2, 'culture': 2, 'lgbtq': 2, 'labor rights': 1}


## 5. Issue: trigger-word shortcuts

Like the students' model, ours still keys on words that co-occur with trigger words. Removing the word flips
each example below.

In [11]:
pd.DataFrame([word_effect(r"\bstation\b", "public services"),
              word_effect(r"\bstudent", "education"),
              word_effect(r"\bpolice\b", "unjust law enforcement"),
              word_effect(r"\bneighborhood\b|\bhouse\b", "housing")]).set_index("word").style.format(PCT)

,class,training rows with word,training: share of class,unknown: predicted with word,unknown: predicted without
word,,,,,
station,public services,1955,72%,49%,14%
student,education,5399,31%,73%,3%
police,unjust law enforcement,4869,16%,26%,1%
neighborhood|house,housing,962,15%,18%,3%


In [12]:
ablations([("FRA14109", [r"\btrain station\b"]), ("DEU131", [r"\bstudent\b"]),
           ("ESP10467", [r"\bpolice\b"]), ("BIH1188", [r"\bhouse\b|\bsettlement\b"])])

- **FRA14109**: foundry workers blocking a train station. Without "train station" the prediction becomes `labor rights` at 0.71.
  72% of training rows with "station" are `public services`, since "railway station" is one of its triggers.
- **DEU131**: a Fridays for Future coal protest becomes `education` because students took part.
- **ESP10467**: police unions negotiating their agreement become `unjust law enforcement` because of the word "police".

## 6. Issue: any war or military word means the Ukraine war

The taxonomy has no class for peace, veterans or the military. The `ukraine-russia war` class absorbs them.

In [13]:
pd.DataFrame([word_effect(r"\bwar\b", "ukraine-russia war"),
              word_effect(r"\bveteran|\bmilitary\b|\bweapon", "ukraine-russia war")]).set_index("word").style.format(PCT)

,class,training rows with word,training: share of class,unknown: predicted with word,unknown: predicted without
word,,,,,
war,ukraine-russia war,1958,63%,15%,1%
veteran|military|weapon,ukraine-russia war,984,43%,18%,1%


In [14]:
ablations([("GBR8215", [r"\bmilitary\b|\bveteran\b|\bconflict\b"]), ("MDA486", [r"\bwar\b"])])

Veterans of the Northern Ireland conflict (GBR8215), and Moldovan veterans demanding a minister's resignation over
a statement about the 1992 war (MDA486), both become Ukraine-war protests. Removing the military words gives
`policies & politics` instead. The word is a strong signal in training, where 63% of rows with "war" are `ukraine-russia war`.

## 7. Issue: ordinary labor disputes end up as `policies & politics` or `public services`

`labor rights` is our weakest large class. Most labor errors have no single word to blame. The model simply prefers
the two catch-all classes.

In [15]:
labor_errors = errors[errors["manual_label"] == "labor rights"]
print(f"labor rights events: {(ev['manual_label'] == 'labor rights').sum()}, lenient errors: {len(labor_errors)}")
print("predicted instead:", labor_errors["predicted_class"].value_counts().to_dict())
print(f"\nunknown events predicted policies & politics: {(unknown['predicted_class'] == 'policies & politics').mean():.0%}, "
      f"public services: {(unknown['predicted_class'] == 'public services').mean():.0%}")

labor rights events: 50, lenient errors: 24
predicted instead: {'policies & politics': 9, 'public services': 6, 'climate': 2, 'unjust law enforcement': 2, 'environment': 1, 'discrimination': 1, 'animal welfare': 1, 'immigration': 1, 'women rights': 1}

unknown events predicted policies & politics: 31%, public services: 15%


In [16]:
ablations([("DEU8787", [r"\bfirefighter\b", r"\bmoney\b"]), ("ITA16435", [r"\bfactory\b"])])

Firefighters demanding a Christmas bonus (DEU8787) and factory workers against their workload (ITA16435) stay
`policies & politics` whichever word is removed. A likely cause is what `labor rights` training data looks like:

In [17]:
spans = (train.groupby("class")["date"].agg(["min", "max"]).apply(lambda s: s.dt.year)
         .rename(columns={"min": "from", "max": "to"}))
spans.loc[["labor rights", "pandemic", "farmers", "climate", "policies & politics", "public services"]]

,from,to
class,,
labor rights,2018,2021
pandemic,2020,2020
farmers,2018,2022
climate,2019,2022
policies & politics,2018,2024
public services,2018,2025


In [18]:
lr = train[train["class"] == "labor rights"]["clean_notes"]
print(f"labor rights training rows: {len(lr)}")
print(f"  from 2020:                          {(train.loc[train['class'] == 'labor rights', 'date'].dt.year == 2020).mean():.0%}")
print(f"  mention covid/pandemic/coronavirus: {lr.str.contains('covid|pandemic|coronavirus').mean():.0%}")
print(f"  mention a strike:                   {lr.str.contains(r'\bstrik', regex=True).mean():.0%}")
print(f"  contain 'suspension' (old 'pension' trigger bug): {lr.str.contains('suspension').sum()}")

labor rights training rows: 3000
  from 2020:                          75%
  mention covid/pandemic/coronavirus: 14%
  mention a strike:                   7%
  contain 'suspension' (old 'pension' trigger bug): 112


The training file takes the **oldest** 3000 rows per class, the opposite of the students' newest-first sample.
`labor rights` therefore covers only 2018–2021, mostly 2020. Only 7% of those rows mention a strike, and 14% are about
Covid. Another 112 contain "suspension", which the old "pension" trigger matched by mistake. That fix never reached
the training file, which was built before it. Such a class teaches the model little about everyday
pay and job disputes. This is a likely cause, but the review cannot prove it.

## 8. What we do better than the students

The students' labels suffer from the "labour conditions" trigger bug, which sends British "labour" to
`animal welfare`. Ours do not:

In [19]:
has = text.str.contains(r"\blabour\b", regex=True)
print(f"unknown events predicted animal welfare: with 'labour' {(unknown.loc[has, 'predicted_class'] == 'animal welfare').mean():.1%}, "
      f"without {(unknown.loc[~has, 'predicted_class'] == 'animal welfare').mean():.1%}")

unknown events predicted animal welfare: with 'labour' 0.0%, without 1.5%


The per-class table in section 1 shows the rest of the trade-off, though several classes have only a handful of
events. Our model is better on climate, public services, health care and unjust law enforcement. The students' model
is better on labor rights, education and culture.

## Summary

| | ours | students |
|---|---|---|
| Strict accuracy | 57.5% | 55.0% |
| Lenient accuracy | 69.0% | 68.5% |
| Counting defensible as right, excluding events with no fitting class | ~81% | ~82% |

Of our 62 lenient errors, 37 are truly wrong, 16 are defensible and 9 fit no class. Most of them are shared
with the students' model. The truly wrong ones come mainly from:

1. **Energy prices read as climate.** Removing "energy" fixes every energy-price example. "Energy" in an unknown
   note means `climate` 58% of the time.
2. **Trigger-word shortcuts.** "station", "student", "police" and "house" decide the topic, as in the students' model.
3. **War and military words mean the Ukraine war.** The taxonomy has no class for peace or veterans.
4. **Labor disputes land in catch-all classes.** The labor training data is 2018–2021, heavy on Covid and polluted
   by the "suspension" bug. This is a likely cause, but not proven.

Two fixes would address several of these at once. Rebuild the training file from the current trigger rules, and
sample it randomly per class instead of taking the oldest rows. The pandemic errors on 2020–2021 events
(FRA4674, DEU6165) remain unexplained.